In [ ]:

#importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_recall_curve
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# Base Model: Linnear Regression & Random Forest
We create 2 models with different algorythms to compare,algorythm is the way the model will learn the data and predict an output after. Chosing a good algoryth is crucial because it will directly impact the efficiency model.
We will test a linnear regression algorythm and a random forest algorythm.
**Linear-regression** models are relatively simple and provide an easy-to-interpret mathematical formula that can generate predictions.
**Random-forest** models combines the output of multiple decision trees to reach a single result
## Step 1: Loading the Data
We import our cleaned_data with all data needed

In [ ]:
df  = pd.read_csv('cleaned_dataset.csv', sep=';')

## Step 2: Defining the Goal (Target) and what is usable (Features)
To train an AI, we must tell it what it needs to guess (the **Target**) and what information it is allowed to use (the **Features**). 
A model will then try to find correlations between the target and the features to build his algorythm.

Then, we clean the table to remove every rows where the value is missing because the Ai can't exploit lines where one value is missing.

In [ ]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We remove rows where the delay is equal to -1 and nan (meaning the data was invalid during cleaning).
df = df[df[target] != -1]
df = df.dropna()

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 3: Translating Text for the Computer (Encoding)
An Artificial Intelligence is a calculating machine: it only understands mathematics. It cannot read words like "Paris" or "Bordeaux". 
We must therefore use a "translator" to convert these station names into numerical codes that the computer can analyze. Encoding of columns where the value isn't usable by the model (strings) into and usable value.
We call it the preprocessor.

In [ ]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=False is used to return a dense array instead of a sparse matrix, which is easier to handle in pipelines.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

## Step 4: Creating the AI "Assembly Line"
We are going to set up our predictions models.
We setup a Linear Regression model and a Random Forest model
We send both model trough the preprocessor to clear unreadable values and then we setup the real algorythm

In [ ]:
# We create a "Pipeline": it is an automated assembly line.
# We create two pipelines: one for the linear regression model and one for the random forest model.
# Linear Regression:
linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])
# Random Forest:
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a random forest algorithm.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])
# Gradient Boosting:
gradient_boosting_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

## Step 5: Creating the final AI
This is the most important step. We will divide our data into two batches:
- **80% for training:** The AI practices guessing the delays and looks at the real answers to learn from its mistakes.
- **20% for testing:** We hide the answers from the AI and ask it to make its predictions to see if it has understood the logic.
And then we train both of our models based on the exact same settings to see which is better.

In [ ]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
# We train both models to compare their performance later.
linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)
gradient_boosting_model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred_lin = linear_model.predict(X_test)
y_pred_forest = forest_model.predict(X_test)
y_pred_gradient_boosting = gradient_boosting_model.predict(X_test)

# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 6: Grading (Performance Evaluation)
Now that the AI has taken its exam and made its predictions, we will compare its answers with reality to give it performance grades.
Then we will compare the result of both of our models to see the better one and improve it later on.

In [ ]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : Linear Regression: {mean_absolute_error(y_test, y_pred_lin):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Random Forest: {mean_absolute_error(y_test, y_pred_forest):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Gradient Boosting: {mean_absolute_error(y_test, y_pred_gradient_boosting):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : Linear Regression: {mean_squared_error(y_test, y_pred_lin):.2f}")
print(f"Mean Squared Error (MSE) : Random Forest: {mean_squared_error(y_test, y_pred_forest):.2f}")
print(f"Mean Squared Error (MSE) : Gradient Boosting: {mean_squared_error(y_test, y_pred_gradient_boosting):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : Linear Regression: {r2_score(y_test, y_pred_lin):.2f}")
print(f"R² Score : Random Forest: {r2_score(y_test, y_pred_forest):.2f}")
print(f"R² Score : Gradient Boosting: {r2_score(y_test, y_pred_gradient_boosting):.2f}")
# We also use a cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_linear_r2 = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_forest_r2 = cross_val_score(forest_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_gradient_boosting_r2 = cross_val_score(gradient_boosting_model, X, y, cv=5, scoring='r2', n_jobs=-1)
score_baseline_r2 = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"\nR² average with Cross-Validation 5 : Linear Regression: {np.mean(scores_linear_r2):.2f} (+/- {np.std(scores_linear_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Random Forest: {np.mean(scores_forest_r2):.2f} (+/- {np.std(scores_forest_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Gradient Boosting: {np.mean(scores_gradient_boosting_r2):.2f} (+/- {np.std(scores_gradient_boosting_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Baseline: {np.mean(score_baseline_r2):.2f} (+/- {np.std(score_baseline_r2):.2f})")
# We also evaluate the baseline model to see how much better our models are compared to just guessing the average delay.
print(f"Mean Absolute Error (MAE) : Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
print(f"Mean Squared Error (MSE) : Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
print(f"R² Score : Baseline: {r2_score(y_test, y_pred_baseline):.2f}")

## Conclusion on the model comparison
We saw that the random forest is better than the linnear regression on every metrics and by a pretty big difference.
This result was expected because the linear regression model is more simple that the random forest, but it don t mean that the linnear regression one is bad, it just baded on basic mathematics formulas making it faster and more efficient but not the best to predict data with the best accuracy.

# Second Model
We saw before that the random forest model is better to predict with more accuracy so we chosed to use one.
Now that we have our aglorythm, we can start to try to improve it.
First we need to try to add more parametters but we need to take in mind that it only use direct or easily deductible parameters. But with what we have now, maybe we could deduce new parameters to give him. For example, if we know the departure and arrival stations, we can do a mean of scheduled train for this trip during this season. It is not the most accurate parameter but it is still better than we don t give any parameter at all.

## Step 1: Adding new parameters
We add 2 new parameters : Number of scheduled trains and number of trains delayed > 15min.

In [72]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of real trains', 'Number of trains delayed > 15min']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 2 : Encoding

In [73]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=False is used to return a dense array instead of a sparse matrix, which is easier to handle in pipelines.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 3: Creating the AI "Assembly Line"

In [74]:
# We create a "Pipeline": it is an automated assembly line.
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

gradient_boosting_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

## Step 4: Creating the final AI

In [75]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
forest_model.fit(X_train, y_train)
linear_model.fit(X_train, y_train)
gradient_boosting_model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_forest_pred = forest_model.predict(X_test)
y_linear_pred = linear_model.predict(X_test)
y_gradient_boosting_pred = gradient_boosting_model.predict(X_test)
# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 5: Grading (Performance Evaluation)

In [76]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : Linear Regression: {mean_absolute_error(y_test, y_linear_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Forest Regression: {mean_absolute_error(y_test, y_forest_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Gradient Boosting: {mean_absolute_error(y_test, y_gradient_boosting_pred):.2f} minutes")
print(f"Mean Absolute Error (MAE) : Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : Linear Regression: {mean_squared_error(y_test, y_linear_pred):.2f}")
print(f"Mean Squared Error (MSE) : Forest Regression: {mean_squared_error(y_test, y_forest_pred):.2f}")
print(f"Mean Squared Error (MSE) : Gradient Boosting: {mean_squared_error(y_test, y_gradient_boosting_pred):.2f}")
print(f"Mean Squared Error (MSE) : Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : Linear Regression: {r2_score(y_test, y_linear_pred):.2f}")
print(f"R² Score : Forest Regression: {r2_score(y_test, y_forest_pred):.2f}")
print(f"R² Score : Gradient Boosting: {r2_score(y_test, y_gradient_boosting_pred):.2f}")
print(f"R² Score : Baseline: {r2_score(y_test, y_pred_baseline):.2f}")
# We use cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_linear_r2 = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_forest_r2 = cross_val_score(forest_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_gradient_boosting_r2 = cross_val_score(gradient_boosting_model, X, y, cv=5, scoring='r2', n_jobs=-1)
score_baseline_r2 = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f"\nR² average with Cross-Validation 5 : Linear Regression: {np.mean(scores_linear_r2):.2f} (+/- {np.std(scores_linear_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Forest Regression: {np.mean(scores_forest_r2):.2f} (+/- {np.std(scores_forest_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Gradient Boosting: {np.mean(scores_gradient_boosting_r2):.2f} (+/- {np.std(scores_gradient_boosting_r2):.2f})")
print(f"\nR² average with Cross-Validation 5 : Baseline: {np.mean(score_baseline_r2):.2f} (+/- {np.std(score_baseline_r2):.2f})")

Mean Absolute Error (MAE) : Linear Regression: 1.63 minutes
Mean Absolute Error (MAE) : Forest Regression: 1.30 minutes
Mean Absolute Error (MAE) : Gradient Boosting: 1.33 minutes
Mean Absolute Error (MAE) : Baseline: 2.96 minutes
Mean Squared Error (MSE) : Linear Regression: 6.32
Mean Squared Error (MSE) : Forest Regression: 6.55
Mean Squared Error (MSE) : Gradient Boosting: 4.33
Mean Squared Error (MSE) : Baseline: 16.61
R² Score : Linear Regression: 0.62
R² Score : Forest Regression: 0.60
R² Score : Gradient Boosting: 0.74
R² Score : Baseline: -0.00

R² average with Cross-Validation 5 : Linear Regression: 0.37 (+/- 0.31)

R² average with Cross-Validation 5 : Forest Regression: 0.39 (+/- 0.32)

R² average with Cross-Validation 5 : Gradient Boosting: 0.53 (+/- 0.28)

R² average with Cross-Validation 5 : Baseline: -0.04 (+/- 0.03)


## Conclusion
We saw that adding new parameters don t necessarly increase the efficiency of the model so we need to think of others things

# Hyperparameters
One solution to improve the model could be the Hyperparamters. Hyperparameters are parameters the guide how the model must learn. In our case we will use Grid Search. Grid Search is an algorythm that will try every combination of hyperparameters to search the best one.

## Step 1 : Setting up the data
We set up everything the same way that before

In [ ]:
target = 'Average delay of all trains at arrival'
#We go back to previous features because we saw that they didn't improve the model
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of scheduled trains', 'Number of trains delayed > 15min']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We create a "Pipeline": it is an automated assembly line.
forest_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Step 2 : We setup the grid search to try differents parameters
We need to initiate which parameters the grid search will use and tru to combine

In [ ]:
# We split the cols into categorical and numeric columns
categorical_cols = ['Service', 'Departure station', 'Arrival station', 'Season']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# We transform the categoricals values into numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# We create a pipeline that first transforms the data and then applies the model
forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

linear_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

gradient_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', HistGradientBoostingRegressor(random_state=42))
])

# We define the grid of hyperparameters to search over
forest_param_grid = {
    # We search for the best number of trees in the random forest
    'model__n_estimators': [125, 150, 275],
    # We search for the best maximum depth of the trees in the random forest
    'model__max_depth': [35, 40, 50, 60],
    # We search for the best minimum number of samples required to split an internal node
    'model__min_samples_split': [4, 5, 8],
    # We search for the best minimum number of samples required to be at a leaf node
    'model__min_samples_leaf': [1, 2, 5],
    # We search for max features to consider when looking for the best split
    'model__max_features': [None, 'sqrt', 'log2']
}

linear_param_grid = {
    # We search for the best fit_intercept parameter for the linear regression
    'model__fit_intercept': [True, False],
    # We search for the best copy_X parameter for the linear regression
    'model__copy_X': [True, False],
}

gradient_param_grid = {
    'regressor__max_iter': [100, 200, 300],
    'regressor__max_leaf_nodes': [31, 50, 100],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__max_depth': [None, 3, 5, 7]
}

# We create the Gridsearch algorythm that will search for the best hyperparameters
forest_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=forest_pipeline, 
    # The dictionary containing the hyperparameters to test
    param_grid=forest_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
    # Uses all available CPU cores to run calculations in parallel
    n_jobs=-1
)

linear_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=linear_pipeline,
    # The dictionary containing the hyperparameters to test
    param_grid=linear_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
     # Uses all available CPU cores to run calculations in parallel
     n_jobs=-1
)

gradient_grid_search = GridSearchCV(
    # The model to tune and evaluate
    estimator=gradient_pipeline,
    # The dictionary containing the hyperparameters to test
    param_grid=gradient_param_grid,
    # Number of cross-validation folds (splits data into 5 parts)
    cv=5,
    # The metric used to evaluate and compare the models' performance
    scoring='r2',
     # Uses all available CPU cores to run calculations in parallel
     n_jobs=-1
)

# We split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 3 : Executing the grid search

Now that everything is setup, we can execute it and see print the results of the grid search 

In [ ]:
# We fit the grid search to the training data
linear_grid_search.fit(X_train, y_train)
forest_grid_search.fit(X_train, y_train)
gradient_grid_search.fit(X_train, y_train)

# We print the best hyperparameters found by the grid search
print("Best hyperparameters :", linear_grid_search.best_params_)
print("Best hyperparameters :", forest_grid_search.best_params_)
print("Best hyperparameters :", gradient_grid_search.best_params_)

We can now create a new model where we apply the hyperparameters and see if it's better.

# Model 3

## Step 1 : initialization of the model
First we init the model like we did before


In [77]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of scheduled trains', 'Number of trains delayed > 15min']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

# We create a "Pipeline": it is an automated assembly line. for the linear regression model
linear_model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a linear algorithm.
    ('regressor', LinearRegression())
])

# We create a "Pipeline": it is an automated assembly line for the random forest model.
model_forest = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# We create a "Pipeline": it is an automated assembly line for the gradient boosting model.
model_gradient_boosting = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features based on a gradient boosting algorithm.
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])



## Step 2 : Apply hyperparameters
Now that we did the setup we can apply the hyperparameters we found to the model

In [78]:
# Apply hyperparameter we found with GridSearchCV
model_forest.set_params(regressor__n_estimators=150)
model_forest.set_params(regressor__max_depth=35)
model_forest.set_params(regressor__min_samples_split=4)
model_forest.set_params(regressor__min_samples_leaf=1)
model_forest.set_params(regressor__max_features=None)
linear_model.set_params(regressor__fit_intercept=True)
linear_model.set_params(regressor__copy_X=True)
model_gradient_boosting.set_params(regressor__max_iter=100)
model_gradient_boosting.set_params(regressor__max_leaf_nodes=50)
model_gradient_boosting.set_params(regressor__learning_rate=0.1)
model_gradient_boosting.set_params(regressor__max_depth=None)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

## Step 3 : Finishing the Ai
We can finally finish the Ai and test it after

In [79]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model_forest.fit(X_train, y_train)
linear_model.fit(X_train, y_train)
model_gradient_boosting.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred_forest = model_forest.predict(X_test)
y_pred_linear = linear_model.predict(X_test)
y_pred_gradient_boosting = model_gradient_boosting.predict(X_test)
# We init a dummy regressor to have a baseline for our models
baseline_model = DummyRegressor(strategy='mean')
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)

## Step 4 : Grading the Ai
We can now grade the model to see if it improved

In [80]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) Random Forest: {mean_absolute_error(y_test, y_pred_forest):.2f} minutes")
print(f"Mean Absolute Error (MAE) Linear Regression: {mean_absolute_error(y_test, y_pred_linear):.2f} minutes")
print(f"Mean Absolute Error (MAE) Gradient Boosting: {mean_absolute_error(y_test, y_pred_gradient_boosting):.2f} minutes")
print(f"Mean Absolute Error (MAE) Baseline: {mean_absolute_error(y_test, y_pred_baseline):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) Random Forest: {mean_squared_error(y_test, y_pred_forest):.2f}")
print(f"Mean Squared Error (MSE) Linear Regression: {mean_squared_error(y_test, y_pred_linear):.2f}")
print(f"Mean Squared Error (MSE) Gradient Boosting: {mean_squared_error(y_test, y_pred_gradient_boosting):.2f}")
print(f"Mean Squared Error (MSE) Baseline: {mean_squared_error(y_test, y_pred_baseline):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score Random Forest: {r2_score(y_test, y_pred_forest):.2f}")
print(f"R² Score Linear Regression: {r2_score(y_test, y_pred_linear):.2f}")
print(f"R² Score Gradient Boosting: {r2_score(y_test, y_pred_gradient_boosting):.2f}")
print(f"R² Score Baseline: {r2_score(y_test, y_pred_baseline):.2f}")
# We use cross-validation to evaluate the model's performance more robustly and to compare each models more precisely.
scores_m3_forest = cross_val_score(model_forest, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_linear = cross_val_score(linear_model, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_gradient_boosting = cross_val_score(model_gradient_boosting, X, y, cv=5, scoring='r2', n_jobs=-1)
scores_m3_baseline = cross_val_score(baseline_model, X, y, cv=5, scoring='r2', n_jobs=-1)

print(f"\nR² average with Cross-Validation 5 Random Forest: {np.mean(scores_m3_forest):.2f} (+/- {np.std(scores_m3_forest):.2f})")
print(f"\nR² average with Cross-Validation 5 Linear Regression: {np.mean(scores_m3_linear):.2f} (+/- {np.std(scores_m3_linear):.2f})")
print(f"\nR² average with Cross-Validation 5 Gradient Boosting: {np.mean(scores_m3_gradient_boosting):.2f} (+/- {np.std(scores_m3_gradient_boosting):.2f})")
print(f"\nR² average with Cross-Validation 5 Baseline: {np.mean(scores_m3_baseline):.2f} (+/- {np.std(scores_m3_baseline):.2f})")


Mean Absolute Error (MAE) Random Forest: 1.34 minutes
Mean Absolute Error (MAE) Linear Regression: 1.65 minutes
Mean Absolute Error (MAE) Gradient Boosting: 1.32 minutes
Mean Absolute Error (MAE) Baseline: 2.96 minutes
Mean Squared Error (MSE) Random Forest: 6.41
Mean Squared Error (MSE) Linear Regression: 6.39
Mean Squared Error (MSE) Gradient Boosting: 4.30
Mean Squared Error (MSE) Baseline: 16.61
R² Score Random Forest: 0.61
R² Score Linear Regression: 0.61
R² Score Gradient Boosting: 0.74
R² Score Baseline: -0.00

R² average with Cross-Validation 5 Random Forest: 0.36 (+/- 0.36)

R² average with Cross-Validation 5 Linear Regression: 0.37 (+/- 0.31)

R² average with Cross-Validation 5 Gradient Boosting: 0.47 (+/- 0.32)

R² average with Cross-Validation 5 Baseline: -0.04 (+/- 0.03)
